# Importar 24DSF BCRA → SQLite → Cloudflare R2

**Pasos:**
1. Subí el archivo `24DSF*.7Z` al panel de archivos de Colab (ícono de carpeta a la izquierda)
2. Ejecutá cada celda en orden
3. Al final la DB queda subida a R2 automáticamente

> **RAM recomendada:** el archivo 24DSF descomprimido puede ser 1-2 GB.
> Usá Runtime → Change runtime type → T4 GPU (o cualquier opción con 12 GB RAM).

In [ ]:
# Celda 1: instalar dependencias
!apt-get install -q p7zip-full
!pip install -q boto3

In [ ]:
# Celda 2: bajar el script de importación desde el repo
!wget -q -O bcra_import_local.py \
  'https://raw.githubusercontent.com/TU_USUARIO/TU_REPO/main/bcra_import_local.py'
# Si el repo es privado o preferís pegar el script manualmente:
# Subilo también al panel de archivos de Colab junto al .7Z

In [ ]:
# Celda 3: configurar credenciales R2
# Pegá los valores de Render/Cloudflare acá (no los commitees al repo)
import os

os.environ['R2_ACCESS_KEY_ID']     = 'TU_ACCESS_KEY'
os.environ['R2_SECRET_ACCESS_KEY'] = 'TU_SECRET_KEY'
os.environ['R2_ENDPOINT_URL']      = 'https://TU_ACCOUNT_ID.r2.cloudflarestorage.com'
os.environ['R2_BUCKET_NAME']       = 'TU_BUCKET'

print('Credenciales R2 cargadas')

In [ ]:
# Celda 4: detectar el archivo 24DSF subido y ejecutar el import
import glob, os

archivos = glob.glob('/content/24DSF*.7Z') + glob.glob('/content/24DSF*.zip') + \
           glob.glob('/content/24dsf*.7z') + glob.glob('/content/24dsf*.zip')

if not archivos:
    print('ERROR: no se encontró el archivo 24DSF*.7Z en /content')
    print('Subilo al panel de archivos de Colab (carpeta izquierda) y volvé a correr esta celda')
else:
    archivo = archivos[0]
    print(f'Archivo detectado: {archivo} ({os.path.getsize(archivo)/1e6:.0f} MB)')
    !python bcra_import_local.py "{archivo}" bcra_nomdeu.db

In [ ]:
# Celda 5 (opcional): verificar la DB generada
import sqlite3

conn = sqlite3.connect('bcra_nomdeu.db')
tablas = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
print('Tablas:', [t[0] for t in tablas])

for tabla in ['denominaciones', 'entidades', 'deudas_resumen', 'historial_detalle']:
    try:
        n = conn.execute(f'SELECT COUNT(*) FROM {tabla}').fetchone()[0]
        print(f'  {tabla}: {n:,} filas')
    except Exception as e:
        print(f'  {tabla}: ERROR — {e}')

# Verificar columnas de historial_detalle
cols = [c[1] for c in conn.execute('PRAGMA table_info(historial_detalle)').fetchall()]
meses = [c for c in cols if c.startswith('sit_')]
print(f'  Meses en historial_detalle: {len(meses)} ({meses[0]} → {meses[-1]})')
conn.close()